# Customer Churn — Feature Engineering and Preprocessing

## Business objective
The goal of this notebook is to convert the EDA findings into a modeling-ready dataset.

## Notebook objectives
This notebook focuses on:
1. creating and justifying derived features,
2. validating the coherence of these engineered features,
3. preparing a reusable preprocessing pipeline,
4. splitting the dataset into train / validation / test sets,
5. producing clean inputs for the modeling phase.

## Scope
This notebook does **not** train predictive models yet.  
Its purpose is to prepare the feature space and preprocessing workflow for downstream modeling.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DB_PATH = Path("../data/processed/churn.duckdb")
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [6]:
conn = duckdb.connect(str(DB_PATH))
df_raw = conn.execute("SELECT * FROM raw_customers").df()
conn.close()

print("raw_customers shape:", df_raw.shape)
df_raw.head()

raw_customers shape: (7043, 21)


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.850,29.850,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.950,"1,889.500",0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.850,108.150,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.300,"1,840.750",0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.700,151.650,1


## 1. Feature engineering strategy

The EDA highlighted several important churn patterns:
- short tenure,
- month-to-month contracts,
- higher monthly charges,
- fiber optic service,
- payment behavior.

These observations motivate the creation of interpretable derived features that may help both predictive performance and business understanding.

In [7]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    service_cols = [
        "online_security",
        "online_backup",
        "device_protection",
        "tech_support",
        "streaming_tv",
        "streaming_movies",
    ]

    # 1. Tenure grouped into interpretable lifecycle stages
    df["tenure_group"] = pd.cut(
        df["tenure"],
        bins=[0, 12, 24, np.inf],
        labels=["new", "mid_term", "long_term"],
        right=False,
        include_lowest=True,
    )

    # 2. Long contract flag
    df["long_contract"] = df["contract"].isin(["One year", "Two year"]).astype(int)

    # 3. High monthly charges flag
    df["high_monthly_charges"] = (df["monthly_charges"] >= 70).astype(int)

    # 4. TotalCharges / tenure proxy
    df["avg_monthly_value_proxy"] = (
        df["total_charges"] / df["tenure"].replace(0, np.nan)
    )

    # 5. Number of subscribed value-added services
    df["subscribed_services_count"] = sum(
        (df[col] == "Yes").astype(int) for col in service_cols
    )

    # 6. Grouped payment behavior
    df["payment_group"] = np.select(
        [
            df["payment_method"].str.contains("automatic", case=False, na=False),
            df["payment_method"].eq("Electronic check"),
            df["payment_method"].eq("Mailed check"),
        ],
        [
            "automatic",
            "electronic_check",
            "mailed_check",
        ],
        default="other",
    )

    # 7. Security / support bundle indicator
    df["has_security_bundle"] = (
        (
            (df["online_security"] == "Yes")
            | (df["device_protection"] == "Yes")
            | (df["tech_support"] == "Yes")
        )
    ).astype(int)

    # 8. Fiber optic customer flag
    df["fiber_optic_customer"] = (df["internet_service"] == "Fiber optic").astype(int)

    return df

In [8]:
df_feat = create_features(df_raw)

print("Feature-engineered dataset shape:", df_feat.shape)

new_columns = [col for col in df_feat.columns if col not in df_raw.columns]
print("Engineered features:", new_columns)

df_feat[new_columns + ["churn"]].head()

Feature-engineered dataset shape: (7043, 29)
Engineered features: ['tenure_group', 'long_contract', 'high_monthly_charges', 'avg_monthly_value_proxy', 'subscribed_services_count', 'payment_group', 'has_security_bundle', 'fiber_optic_customer']


,tenure_group,long_contract,high_monthly_charges,avg_monthly_value_proxy,subscribed_services_count,payment_group,has_security_bundle,fiber_optic_customer,churn
0,new,0,0,29.850,1,electronic_check,0,0,0
1,long_term,1,0,55.574,2,mailed_check,1,0,0
2,new,0,0,54.075,2,mailed_check,1,0,1
3,long_term,1,0,40.906,3,automatic,1,0,0
4,new,0,1,75.825,0,electronic_check,0,1,1


## 2. Feature catalog and justification

The following engineered features are created in this notebook:

- `tenure_group`: captures customer lifecycle stage.
- `long_contract`: summarizes contractual commitment.
- `high_monthly_charges`: captures a potentially high billing threshold.
- `avg_monthly_value_proxy`: approximates monthly accumulated value.
- `subscribed_services_count`: measures product/service adoption intensity.
- `payment_group`: reduces payment method granularity into business-relevant groups.
- `has_security_bundle`: captures protective/support service adoption.
- `fiber_optic_customer`: isolates a segment that showed elevated churn risk in EDA.

This exceeds the minimum requirement of at least 5 derived features.

In [9]:
feature_catalog = pd.DataFrame(
    {
        "feature": [
            "tenure_group",
            "long_contract",
            "high_monthly_charges",
            "avg_monthly_value_proxy",
            "subscribed_services_count",
            "payment_group",
            "has_security_bundle",
            "fiber_optic_customer",
        ],
        "type": [
            "categorical",
            "binary",
            "binary",
            "numeric",
            "numeric",
            "categorical",
            "binary",
            "binary",
        ],
        "business_rationale": [
            "Customer lifecycle stage may strongly influence churn risk.",
            "Longer commitments generally reduce churn probability.",
            "High monthly bills may reflect price sensitivity or dissatisfaction.",
            "Approximates accumulated monthly value using billing history.",
            "More subscribed services may indicate deeper product adoption.",
            "Groups payment methods into more interpretable behavioral buckets.",
            "Security/support services may improve retention and stickiness.",
            "Fiber optic customers showed higher churn in EDA and deserve isolation.",
        ],
    }
)

feature_catalog

,feature,type,business_rationale
0,tenure_group,categorical,Customer lifecycle stage may strongly influenc...
1,long_contract,binary,Longer commitments generally reduce churn prob...
2,high_monthly_charges,binary,High monthly bills may reflect price sensitivi...
3,avg_monthly_value_proxy,numeric,Approximates accumulated monthly value using b...
4,subscribed_services_count,numeric,More subscribed services may indicate deeper p...
5,payment_group,categorical,Groups payment methods into more interpretable...
6,has_security_bundle,binary,Security/support services may improve retentio...
7,fiber_optic_customer,binary,Fiber optic customers showed higher churn in E...


## 3. Missing values after feature engineering

At this stage, missing values are intentionally **not imputed yet**.

This is important because imputation should be learned on the **training set only** to avoid data leakage.  
Therefore, missing values will be handled inside the reusable scikit-learn preprocessing pipeline.

In [10]:
missing_after_fe = (
    df_feat.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

missing_after_fe["missing_pct"] = (missing_after_fe["missing_count"] / len(df_feat) * 100).round(3)
missing_after_fe[missing_after_fe["missing_count"] > 0]

,missing_count,missing_pct
total_charges,11,0.156
avg_monthly_value_proxy,11,0.156


## 4. Quick validation of engineered features

Before building the preprocessing pipeline, we verify that the engineered features behave coherently and remain aligned with churn intuition from the EDA.

In [23]:
def churn_rate_table(df: pd.DataFrame, column: str) -> pd.DataFrame:
    table = (
        df.groupby(column, observed=False)
        .agg(
            customers=("churn", "size"),
            churn_rate=("churn", "mean")
        )
        .reset_index()
        .sort_values("churn_rate", ascending=False)
    )
    table["churn_rate_pct"] = (table["churn_rate"] * 100).round(2)
    return table

In [24]:
engineered_feature_checks = {
    "tenure_group": churn_rate_table(df_feat, "tenure_group"),
    "long_contract": churn_rate_table(df_feat, "long_contract"),
    "high_monthly_charges": churn_rate_table(df_feat, "high_monthly_charges"),
    "payment_group": churn_rate_table(df_feat, "payment_group"),
    "has_security_bundle": churn_rate_table(df_feat, "has_security_bundle"),
    "fiber_optic_customer": churn_rate_table(df_feat, "fiber_optic_customer"),
}

for name, table in engineered_feature_checks.items():
    print(f"\n=== {name.upper()} ===")
    display(table)


=== TENURE_GROUP ===


,tenure_group,customers,churn_rate,churn_rate_pct
0,new,2069,0.483,48.280
1,mid_term,1047,0.295,29.510
2,long_term,3927,0.143,14.290



=== LONG_CONTRACT ===


,long_contract,customers,churn_rate,churn_rate_pct
0,0,3875,0.427,42.710
1,1,3168,0.068,6.760



=== HIGH_MONTHLY_CHARGES ===


...

,payment_group,customers,churn_rate,churn_rate_pct
1,electronic_check,2365,0.453,45.290
2,mailed_check,1612,0.191,19.110
0,automatic,3066,0.160,15.980



=== HAS_SECURITY_BUNDLE ===


,has_security_bundle,customers,churn_rate,churn_rate_pct
0,0,3264,0.315,31.460
1,1,3779,0.223,22.280



=== FIBER_OPTIC_CUSTOMER ===


,fiber_optic_customer,customers,churn_rate,churn_rate_pct
1,1,3096,0.419,41.890
0,0,3947,0.145,14.490


## 5. Build the modeling dataset

We now define:
- the target,
- the identifier to exclude,
- the final input matrix before preprocessing.

In [13]:
TARGET_COL = "churn"
ID_COL = "customer_id"

feature_columns = [col for col in df_feat.columns if col not in [TARGET_COL, ID_COL]]

X = df_feat[feature_columns].copy()
y = df_feat[TARGET_COL].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of input features before preprocessing:", len(feature_columns))

X shape: (7043, 27)
y shape: (7043,)
Number of input features before preprocessing: 27


## 6. Train / validation / test split

We use a stratified split in order to preserve the churn distribution across subsets.

Chosen split:
- 70% train
- 15% validation
- 15% test

In [14]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.17647058823529413,  # gives ~15% of total for validation
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_valid.shape, y_valid.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (4929, 27) (4929,)
Validation shape: (1057, 27) (1057,)
Test shape: (1057, 27) (1057,)


In [15]:
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train), len(X_valid), len(X_test)],
        "churn_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
    }
)

split_summary["rows_pct"] = (split_summary["rows"] / len(df_feat) * 100).round(2)
split_summary["churn_rate_pct"] = (split_summary["churn_rate"] * 100).round(2)

split_summary

,split,rows,churn_rate,rows_pct,churn_rate_pct
0,train,4929,0.265,69.980,26.540
1,validation,1057,0.266,15.010,26.580
2,test,1057,0.265,15.010,26.490


## 7. Define preprocessing logic

The preprocessing pipeline will include:
- numeric imputation with median,
- categorical imputation with most frequent value,
- standardization of numeric features,
- one-hot encoding of categorical features.

This makes the workflow reusable and compatible with downstream modeling pipelines.

In [16]:
numeric_features = [
    "senior_citizen",
    "tenure",
    "monthly_charges",
    "total_charges",
    "avg_monthly_value_proxy",
    "subscribed_services_count",
    "long_contract",
    "high_monthly_charges",
    "has_security_bundle",
    "fiber_optic_customer",
]

categorical_features = [
    "gender",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method",
    "tenure_group",
    "payment_group",
]

unused_features = set(X.columns) - set(numeric_features) - set(categorical_features)
assert len(unused_features) == 0, f"Unassigned features found: {unused_features}"

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 10
Categorical features: 17


In [22]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 8. Fit the preprocessor on the training set only

To avoid leakage:
- the preprocessing pipeline is fitted on the **training set**,
- then applied to validation and test sets.

In [18]:
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

feature_names_after_preprocessing = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names_after_preprocessing,
    index=X_train.index,
)

X_valid_processed_df = pd.DataFrame(
    X_valid_processed,
    columns=feature_names_after_preprocessing,
    index=X_valid.index,
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names_after_preprocessing,
    index=X_test.index,
)

print("Processed train shape:", X_train_processed_df.shape)
print("Processed validation shape:", X_valid_processed_df.shape)
print("Processed test shape:", X_test_processed_df.shape)
print("Number of features after preprocessing:", len(feature_names_after_preprocessing))

Processed train shape: (4929, 57)
Processed validation shape: (1057, 57)
Processed test shape: (1057, 57)
Number of features after preprocessing: 57


In [19]:
processed_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train_processed_df), len(X_valid_processed_df), len(X_test_processed_df)],
        "columns_after_preprocessing": [
            X_train_processed_df.shape[1],
            X_valid_processed_df.shape[1],
            X_test_processed_df.shape[1],
        ],
        "missing_values_after_preprocessing": [
            int(X_train_processed_df.isna().sum().sum()),
            int(X_valid_processed_df.isna().sum().sum()),
            int(X_test_processed_df.isna().sum().sum()),
        ],
    }
)

processed_summary

,split,rows,columns_after_preprocessing,missing_values_after_preprocessing
0,train,4929,57,0
1,validation,1057,57,0
2,test,1057,57,0


In [20]:
X_train_processed_df.head()

,num__senior_citizen,num__tenure,num__monthly_charges,num__total_charges,num__avg_monthly_value_proxy,num__subscribed_services_count,num__long_contract,num__high_monthly_charges,num__has_security_bundle,num__fiber_optic_customer,cat__gender_Female,cat__gender_Male,cat__partner_No,cat__partner_Yes,cat__dependents_No,cat__dependents_Yes,cat__phone_service_No,cat__phone_service_Yes,cat__multiple_lines_No,cat__multiple_lines_No phone service,cat__multiple_lines_Yes,cat__internet_service_DSL,cat__internet_service_Fiber optic,cat__internet_service_No,cat__online_security_No,cat__online_security_No internet service,cat__online_security_Yes,cat__online_backup_No,cat__online_backup_No internet service,cat__online_backup_Yes,cat__device_protection_No,cat__device_protection_No internet service,cat__device_protection_Yes,cat__tech_support_No,cat__tech_support_No internet service,cat__tech_support_Yes,cat__streaming_tv_No,cat__streaming_tv_No internet service,cat__streaming_tv_Yes,cat__streaming_movies_No,cat__streaming_movies_No internet service,cat__streaming_movies_Yes,cat__contract_Month-to-month,cat__contract_One year,cat__contract_Two year,cat__paperless_billing_No,cat__paperless_billing_Yes,cat__payment_method_Bank transfer (automatic),cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__tenure_group_long_term,cat__tenure_group_mid_term,cat__tenure_group_new,cat__payment_group_automatic,cat__payment_group_electronic_check,cat__payment_group_mailed_check
4926,-0.434,1.196,-0.151,0.642,-0.141,0.511,1.104,-1.022,0.930,-0.881,0.000,1.000,0.000,1.000,1.000,0.000,0.000,1.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,1.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,1.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000
6392,-0.434,1.115,-0.495,0.342,-0.450,0.511,-0.906,-1.022,-1.076,-0.881,1.000,0.000,0.000,1.000,0.000,1.000,1.000,0.000,0.000,1.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,1.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,1.000,1.000,0.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,1.000,0.000,0.000,0.000,1.000,0.000
6890,-0.434,0.831,-0.120,0.469,-0.048,-0.026,1.104,-1.022,0.930,-0.881,0.000,1.000,1.000,0.000,1.000,0.000,0.000,1.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,1.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,1.000,0.000,0.000,0.000,1.000,0.000
3194,2.303,0.427,-0.303,0.022,-0.341,1.049,-0.906,-1.022,0.930,-0.881,1.000,0.000,0.000,1.000,1.000,0.000,1.000,0.000,0.000,1.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,1.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,1.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000
2599,-0.434,1.600,0.273,1.306,0.275,1.586,1.104,0.979,0.930,-0.881,1.000,0.000,1.000,0.000,1.000,0.000,0.000,1.000,1.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,1.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000


## 9. Save useful preprocessing artifacts

At this stage, we save:
- the fitted preprocessor,
- the engineered train / validation / test datasets (before encoding),
- the processed target vectors.

These artifacts make the workflow easier to reuse in the modeling notebook and later in scripts.

In [21]:
train_engineered = X_train.copy()
train_engineered[TARGET_COL] = y_train.values

valid_engineered = X_valid.copy()
valid_engineered[TARGET_COL] = y_valid.values

test_engineered = X_test.copy()
test_engineered[TARGET_COL] = y_test.values

train_engineered.to_csv(PROCESSED_DIR / "train_engineered.csv", index=False)
valid_engineered.to_csv(PROCESSED_DIR / "validation_engineered.csv", index=False)
test_engineered.to_csv(PROCESSED_DIR / "test_engineered.csv", index=False)

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

print("Saved:")
print("-", PROCESSED_DIR / "train_engineered.csv")
print("-", PROCESSED_DIR / "validation_engineered.csv")
print("-", PROCESSED_DIR / "test_engineered.csv")
print("-", MODELS_DIR / "preprocessor.joblib")

Saved:
- ../data/processed/train_engineered.csv
- ../data/processed/validation_engineered.csv
- ../data/processed/test_engineered.csv
- ../models/preprocessor.joblib


The preprocessing workflow is now fully defined and reusable. The engineered dataset, fitted preprocessor, and train/validation/test splits are ready for model training and comparison.

## 10. Key takeaways

### Main outcomes
1. A modeling-ready feature space has been created from the raw customer table.
2. Eight engineered features were added, exceeding the minimum project requirement.
3. Missing values were intentionally left for pipeline-based imputation in order to avoid leakage.
4. The dataset was split into train / validation / test subsets using stratification.
5. A reusable preprocessing pipeline was built with numeric imputation, scaling, and categorical one-hot encoding.
6. The fitted preprocessor and engineered splits were saved for downstream reuse.

### Why this matters
This notebook transforms EDA insights into reproducible analytical assets:
- interpretable business features,
- a leakage-safe preprocessing workflow,
- and clean inputs for the modeling phase.

### Next step
The next notebook will focus on model training and comparison:
- baseline logistic regression,
- tree-based models,
- gradient boosting,
- and evaluation using churn-oriented metrics.